


# READ ME
## Important
### When you use this script, do a Ctrl+F search for .csv and change the path and file name according to the input CSV file.
## Purpose of the script
### Take raw data and calculate the average according to food type (high or low caloric density)

This script serves for gut_hormones_data_prep.ipynb


In [13]:
import pandas as pd
import numpy as np


# Step 1: Read the CSV file into a DataFrame with low_memory=False to handle mixed types - changer le CSV au besoin
file_path = 'C:\\Users\\Patrick\\Gutbrain_R\\data\\final_merged_quest.csv'
df = pd.read_csv(file_path, low_memory=False)





In [14]:
filtered_df = df[df['onset'].isna()]

In [15]:
# Step 2: Group by 'id_participant', 'session', and 'calories' and calculate mean and standard deviation for 'value' and 'response_time'
# Step 2: Group by 'id_participant', 'session', and 'calories' and calculate mean and standard deviation for all numerical columns
numerical_cols = df.select_dtypes(include='number').columns
agg_dict = {col: ['mean', 'std'] for col in numerical_cols}

aggregated = df.groupby(['id_participant', 'session', 'calories']).agg(agg_dict).reset_index()

# Flatten the MultiIndex columns
aggregated.columns = ['_'.join(col).strip('_') for col in aggregated.columns.values]

In [16]:
# Remove columns that include '_std' in their label
aggregated = aggregated.loc[:, ~aggregated.columns.str.contains('_std')]

# Remove the string '_mean' from all labels, but not if the label has two '_mean' substrings
aggregated.columns = aggregated.columns.str.replace(r'(?<!_mean)_mean(?!_mean)', '', regex=True)
# Remove one occurrence of '_mean' if the label has two '_mean' substrings
aggregated.columns = aggregated.columns.str.replace(r'(_mean.*)_mean', r'\1', regex=True)

In [17]:
# Get the column names of both dataframes
filtered_df_columns = set(filtered_df.columns)
aggregated_columns = set(aggregated.columns)

# Find the columns that are in one dataframe but not the other
different_columns = filtered_df_columns.symmetric_difference(aggregated_columns)



In [18]:
filtered_df = filtered_df.drop(columns=different_columns)

In [19]:
aggregated = aggregated.loc[:, ~aggregated.columns.duplicated()]


In [20]:
filtered_df = filtered_df[list(aggregated.columns)]

In [21]:
# Find equal combinations of 'session' and 'id_participant' between aggregated_columns and filtered_df
common_combinations = aggregated[aggregated[['session', 'id_participant']].apply(tuple, axis=1).isin(
    filtered_df[['session', 'id_participant']].apply(tuple, axis=1))]



In [22]:


# Group by 'id_participant' and filter groups where all four sessions are present
participants_with_all_sessions = filtered_df.groupby('id_participant').filter(lambda x: x['session'].nunique() == 4)



# Remove participants with all four sessions
filtered_df_without_all_sessions = filtered_df[~filtered_df['id_participant'].isin(participants_with_all_sessions['id_participant'].unique())]

# Group by 'id_participant' and filter groups where exactly three sessions are present
participants_with_three_sessions = filtered_df_without_all_sessions.groupby('id_participant').filter(lambda x: x['session'].nunique() == 3)



# Display the participants with exactly three sessions
participants_with_three_sessions

# Only keep rows where age is not NaN
filtered_df_without_all_sessions_wo_age = filtered_df_without_all_sessions.dropna(subset=['age'])




In [23]:
avg_bid_and_Nan = pd.concat([aggregated, filtered_df_without_all_sessions_wo_age], axis=0)


In [24]:
# Step 7: Export the resulting DataFrame to a new CSV file - changer le nom du csv au besoin
output_file_path = 'C:\\Users\\Patrick\\Gutbrain_R\\data\\avg_bid_and_Nan.csv'
avg_bid_and_Nan.to_csv(output_file_path, index=False)